# 04 - Paragraph chunking

Paragraph chunking follows document layout. It first splits text into paragraphs, then keeps each paragraph together when possible.

In RAG, paragraph chunks often work well because a paragraph usually contains a complete idea: claim, explanation, evidence, or decision reasoning.

In [ ]:
from pathlib import Path
import re

from pypdf import PdfReader

ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / "data/sample_documents/AUG242026_05B2203_AAO.pdf").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Start Jupyter inside the Awesome_RAG checkout")
PDF_PATH = ROOT / "data" / "sample_documents" / "AUG242026_05B2203_AAO.pdf"

def normalize_text(text):
    text = text.replace("\r\n", "\n").replace("\r", "\n")
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()

def load_pdf_pages(path):
    reader = PdfReader(str(path))
    pages = []
    for page_number, page in enumerate(reader.pages, start=1):
        text = normalize_text(page.extract_text() or "")
        if text:
            pages.append({"page": page_number, "text": text})
    return pages

def chunk_stats(chunks):
    lengths = [len(chunk["text"].split()) for chunk in chunks]
    return {"chunks": len(chunks), "min_words": min(lengths), "avg_words": round(sum(lengths) / len(lengths), 1), "max_words": max(lengths)}

def print_chunks(chunks, limit=5, preview_chars=650):
    for chunk in chunks[:limit]:
        print(f"{chunk['id']} | page {chunk['page_start']}-{chunk['page_end']}")
        print(chunk["text"][:preview_chars])
        print("-" * 100)
    if len(chunks) > limit:
        print(f"... {len(chunks) - limit} more chunks")

pages = load_pdf_pages(PDF_PATH)
{"pages": len(pages), "words": sum(len(page["text"].split()) for page in pages)}

## Paragraph chunking function

`re.split(r"\n\s*\n", page["text"])` splits where there is a blank line. That is how we detect paragraphs from extracted PDF text.

If a paragraph has 220 words or fewer, it becomes one chunk.

If a paragraph has more than 220 words, it is split into multiple chunks from the same paragraph. It does not become a new paragraph; it becomes `paragraph-1`, `paragraph-2`, etc. because one natural paragraph was too large for the chunk budget.

In [ ]:
def paragraph_chunks(pages, max_words=220):
    """Keep extracted paragraphs together unless a paragraph exceeds max_words."""
    chunks = []

    for page in pages:
        paragraphs = [part.strip() for part in re.split(r"\n\s*\n", page["text"]) if part.strip()]

        for paragraph in paragraphs:
            words = paragraph.split()

            if len(words) <= max_words:
                chunks.append({
                    "id": f"paragraph-{len(chunks) + 1}",
                    "page_start": page["page"],
                    "page_end": page["page"],
                    "text": paragraph,
                })
            else:
                for start in range(0, len(words), max_words):
                    end = min(start + max_words, len(words))
                    chunks.append({
                        "id": f"paragraph-{len(chunks) + 1}",
                        "page_start": page["page"],
                        "page_end": page["page"],
                        "text": " ".join(words[start:end]),
                    })

    return chunks

chunks = paragraph_chunks(pages, max_words=220)
chunk_stats(chunks)

In [ ]:
print_chunks(chunks, limit=6)

## Sentence vs paragraph chunking

Sentence chunking follows grammar. Paragraph chunking follows layout.

A sentence chunk can split one paragraph into several chunks. A paragraph chunk tries to keep the whole paragraph together. For legal and policy documents, paragraph chunking often gives the model more useful context, but it depends on whether PDF extraction preserved paragraph breaks correctly.